# CEOPRO AI Chat

Run the cells in order, top to bottom:
1. **Install** - installs every package this needs. Run once only.
2. **Connect** - asks for your Groq key, connects to the database.
3. **Chat** - type your questions here.

In [ ]:
%pip install -q torch>=2.0,<3.0 transformers>=4.40,<6.0 sentence-transformers>=3.0,<4.0 faiss-cpu>=1.8,<2.0 rank-bm25>=0.2,<0.3 httpx>=0.27,<1.0 psycopg2-binary>=2.9,<3.0 minio>=7.2,<8.0 xgboost>=2.0,<3.0 pandas>=2.0,<3.0 numpy>=1.24,<2.0 scikit-learn>=1.3,<2.0 redis>=5.0,<6.0 pdfplumber>=0.11,<0.12 openpyxl>=3.1,<4.0 python-docx>=1.1,<2.0 accelerate>=1.1.0,<2.0 fastapi>=0.110,<1.0 uvicorn>=0.29,<1.0 PyJWT>=2.8,<3.0 python-multipart>=0.0.9,<0.1 sentencepiece>=0.2,<0.3 protobuf>=4.25,<8.0 psutil>=5.9,<8.0 python-dotenv groq

import sys
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
if sys.version_info < (3, 9):
    print("WARNING: this project is tested on Python 3.11+ - below 3.9, some packages above may not install cleanly.")
print("Dependencies installed. Run this cell once - then run the cells below (safe to re-run them without repeating this one).")

In [ ]:
import getpass
import os
import sys
import time

from dotenv import load_dotenv

REPO_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(REPO_ROOT, "src", "ai")):
    REPO_ROOT = r"C:\CEO PRO\CEOPRO-AI\.claude\worktrees\competitors-market-sentiment-a0ba02"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Loads the disposable local test stack's connection details from .env at
# the repo root (gitignored - never committed). No credential values live
# in this notebook's own saved source at all - if .env is missing or
# incomplete, this fails loudly rather than silently falling back to a
# hardcoded value.
load_dotenv(os.path.join(REPO_ROOT, ".env"))
_required = ["DATABASE_URL", "APP_DB_PASSWORD", "MINIO_ENDPOINT", "MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD"]
_missing = [v for v in _required if not os.getenv(v)]
if _missing:
    raise RuntimeError(
        f"Missing from .env: {', '.join(_missing)}. Create a .env file at the repo root "
        "with these variables - see .env.example for the format."
    )

os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your GROQ_API_KEY (hidden): ") or ""

TENANT_ID = "68bf13ff-1782-4814-a7fa-1d370ae1c223"
USER_ID = "8cd536f2-a9f9-464d-98b0-69a27852e978"

from src.ai import db
from src.ai.rag import embeddings as rag_embeddings
from src.ai.rag import llm_client as rag_llm_client
from src.ai.rag import pipeline as rag_pipeline
from src.ai.rag import reranking as rag_reranking

conn = db.app_role_connection(TENANT_ID, USER_ID)

# Pre-warm everything that's otherwise loaded lazily on the FIRST chat
# question - the embedding model, the Cross-Encoder reranker, and this
# tenant's hybrid (BM25 + FAISS) index. Without this, whichever question
# you type first in the Chat cell eats a one-time ~20-40s model-load cost
# on top of the actual retrieval - doing it here means every question in
# the Chat cell is fast, not just the second one onward.
print("Warming up models (one-time, ~20-40s)...")
_warm_start = time.time()
rag_embeddings.get_model()
rag_reranking.get_model()
rag_pipeline.build_hybrid_index(conn, TENANT_ID)
print(f"Warm-up done in {time.time() - _warm_start:.1f}s.")

print("Ready. Run the next cell to chat.")

In [ ]:
import time

# rerank_candidates=10 instead of the library default of 20: the
# Cross-Encoder reranker does one forward pass per candidate (see
# reranking.py's own docstring), so this halves that stage's cost. Only
# tuned here, for this interactive test notebook - the shared
# run_retrieval()/answer_query() default (20) is untouched for every
# other caller, since a smaller pool is a real recall/precision trade,
# not a free win.
RERANK_CANDIDATES = 10

while True:
    query_text = input("You: ").strip()
    if not query_text:
        continue
    if query_text.lower() in ("exit", "quit"):
        print("Stopped. Re-run this cell to chat again.")
        break
    _t0 = time.time()
    try:
        result = rag_llm_client.answer_query(
            conn, TENANT_ID, query_text, top_k=5, rerank_candidates=RERANK_CANDIDATES,
        )
        conn.commit()
    except rag_llm_client.LLMError as e:
        conn.rollback()
        print(f"LLM call failed: {e}\n")
        continue
    except Exception as e:
        conn.rollback()
        print(f"Query failed: {e}\n")
        continue
    print(f"\nAssistant: {result['answer']}\n")
    if result["sources"]:
        print("Sources:")
        for source in result["sources"]:
            print(f"  [{source['source_index']}] chunk={source['chunk_id']} score={source['score']}")
    print(f"({time.time() - _t0:.1f}s)\n")